# Output-calibration VALIDATION

Validates the **applied** sensor cal (derived from the phased ≥5-run calibration) on
**independent** data — never the calibration set:

1. **Multisine maneuver** (1 run) — all axes excited together at distinct frequencies, so it
   tests the cal's cross-axis **coupling** terms that phased single-axis derivation can't self-check.
2. **Landing to touchdown** (1 run) — the real operating regime (descent, low alt, marker→ring handoff).

Both compare the **calibrated** corner + ring flow against **Gazebo GT** (GT-direct only; the
corner-as-transfer-standard trick is a derivation expedient, not a validation). The cal is
auto-loaded from `src/img_data.py`, so this always reflects what the runtime actually applies.

> 1 run is enough to validate the **signal** (R²/tracking). Landing **performance** (xy/softness)
> is a different question that needs n≥5.

In [ ]:
import os, sys, glob, numpy as np, matplotlib.pyplot as plt, datetime as _dt
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'tools'))
import validate_flow_to_touchdown as vft   # gt_v_flow, best_offset, validate, CAL, CAL_RING

# Pick which validation runs to load (0 = latest, 1 = previous, ...). The picker
# lists ALL runs in each folder so you can access any of them; set the *_INDEX.
MULTISINE_INDEX = 0
LANDING_INDEX   = 0
N_SHOW = 25

def _list_runs(pats):
    runs = {}
    for pat in pats:
        for d in glob.glob(pat):
            if os.path.isdir(d) and os.path.isfile(os.path.join(d, 'Img_Data.npy')):
                runs[d] = os.path.getmtime(d)
    return sorted(runs.items(), key=lambda kv: -kv[1])      # most recent first

def _pick(name, primary_pats, fallback_pats, index):
    runs = _list_runs(primary_pats)
    src = 'validation_data'
    if not runs:                                   # only fall back when no validation run exists
        runs = _list_runs(fallback_pats); src = 'fallback (NOT independent)'
    print(f'{name} runs [{src}] (most recent first; pick via {name}_INDEX):')
    for i, (d, mt) in enumerate(runs[:N_SHOW]):
        mark = '  <-- CURRENT' if i == index else ''
        print(f'  [{i:2d}] {_dt.datetime.fromtimestamp(mt):%Y-%m-%d %H:%M:%S}  {os.path.basename(d)}{mark}')
    if len(runs) > N_SHOW: print(f'  ... and {len(runs) - N_SHOW} older')
    if not runs: print('  (none found)'); return None
    return runs[min(index, len(runs) - 1)][0]

MULTISINE_RUN = _pick('MULTISINE', ['../validation_data/multisine/*'],
                      ['../calibration_data/output/*'], MULTISINE_INDEX)
print()
LANDING_RUN   = _pick('LANDING', ['../validation_data/landing/*'],
                      ['../test_data/Landing_Test/*', '../test_data/RingFlow/*'], LANDING_INDEX)
print('\nMULTISINE_RUN =', MULTISINE_RUN)
print('LANDING_RUN   =', LANDING_RUN)

In [ ]:
# Applied cal (auto-loaded from src/img_data.py via the validate tool)
print('corner _sensor_cal_hw diag :', np.round(np.diag(vft.CAL), 4))
print('ring   _sensor_cal_ring diag:', np.round(np.diag(vft.CAL_RING), 4),
      '(identity =', np.allclose(vft.CAL_RING, np.eye(6)), ')')

def r2(a, b):
    m = np.isfinite(a) & np.isfinite(b)
    if m.sum() < 8: return np.nan
    return 1 - np.sum((a[m]-b[m])**2) / (np.sum((b[m]-b[m].mean())**2) + 1e-12)
CHN = ['h_x', 'h_y', 'h_z', 'w_x', 'w_y', 'w_z']

## 1 — Multisine validation: calibrated corner + ring flow vs GT

Per-channel R² annotated. The cal was derived from *phased* runs, so good R² here (different,
all-axes-at-once excitation) is genuine **generalization**, and specifically exercises the
off-diagonal h↔w coupling.

In [ ]:
d = MULTISINE_RUN
img = np.load(d + '/Img_Data.npy', allow_pickle=True).item()
gt  = np.load(d + '/Ground_Truth.npy', allow_pickle=True).item()
t_g, Vh, Vw, Vz = vft.gt_v_flow(gt)
GT = np.hstack([Vh, -Vw])                                   # manuscript w = -V_w_ug

ti   = np.asarray(img['Time'], float)
corn = np.asarray(img['Opt Flow Ang Vel'], float)
ring = np.asarray(img.get('Ring Opt Flow Ang Vel', np.zeros((len(ti), 6))), float)
ncc  = np.asarray(img['N Flow Corners'], float)
ncr  = np.asarray(img.get('N Ring Corners', np.zeros(len(ti))), float)
n = min(len(ti), len(corn), len(ring), len(ncc), len(ncr))
ti, corn, ring, ncc, ncr = ti[:n], corn[:n], ring[:n], ncc[:n], ncr[:n]

corn_cal = (vft.CAL @ corn.T).T                            # runtime-faithful corner
ring_cal = (vft.CAL_RING @ ring.T).T                       # runtime-faithful ring
idiv = np.where(ncc > 0, corn_cal[:, 2], np.nan)
off, ar = vft.best_offset(ti, idiv, t_g, GT[:, 2])
ti_r, tg_r = ti - ti[0], t_g - t_g[0]
GTi = np.column_stack([np.interp(ti_r - off, tg_r, GT[:, k], left=np.nan, right=np.nan) for k in range(6)])

fig, axes = plt.subplots(6, 1, figsize=(12, 14), sharex=True, constrained_layout=True)
for i, name in enumerate(CHN):
    ax = axes[i]
    ax.plot(ti_r, GTi[:, i], 'k--', lw=1.3, alpha=0.85, label='Ground truth')
    cm = ncc > 0; ax.plot(ti_r[cm], corn_cal[cm, i], 'C0', lw=0.9, label='corner (cal)')
    rm = ncr > 0; ax.plot(ti_r[rm], ring_cal[rm, i], 'C3', lw=1.0, alpha=0.8, label='ring (cal)')
    ax.set_ylabel(name)
    ax.set_title(f'corner R2={r2(corn_cal[:, i], GTi[:, i]):.2f}   ring R2={r2(ring_cal[:, i], GTi[:, i]):.2f}',
                 fontsize=8, loc='right')
    if i == 0: ax.legend(fontsize=8, loc='upper right')
axes[-1].set_xlabel('t (s, relative)')
fig.suptitle(f'MULTISINE validation — calibrated flow vs GT  (align r={ar:.2f}, off={off:+.1f}s)', fontsize=12)
plt.show()

## EKF FUSED optical flow (corner+ring) — target-relative `[h_tr; w]`

Needs a `FLOW_FUSE_RING=1` recording (populates `Opt Flow Fused` + `Target Vel`). Shows the EKF's
target-relative flow vs corner vs GT, plus the estimated target velocity `h_tv` (≈0 stationary; =
rover velocity if moving).

In [ ]:
fused = np.asarray(img.get('Opt Flow Fused', []), float)
tvel  = np.asarray(img.get('Target Vel', []), float)
if fused.ndim != 2 or len(fused) == 0 or not np.any(fused):
    print("No EKF fused data — record with FLOW_FUSE_RING=1 to populate 'Opt Flow Fused'/'Target Vel'.")
else:
    nf = min(len(fused), len(ti)); tf = ti[:nf] - ti[0]
    fGT = np.column_stack([np.interp(tf - off, t_g - t_g[0], GT[:, k], left=np.nan, right=np.nan) for k in range(6)])
    fig, axes = plt.subplots(6, 1, figsize=(12, 14), sharex=True, constrained_layout=True)
    for i, name in enumerate(CHN):
        ax = axes[i]
        ax.plot(tf, fGT[:, i], 'k--', lw=1.3, alpha=0.85, label='GT')
        ax.plot((ti - ti[0])[:nf], corn_cal[:nf, i], 'C0', lw=0.7, alpha=0.5, label='corner (cal)')
        ax.plot(tf, fused[:nf, i], 'C2', lw=1.1, label='EKF fused h_tr')
        ax.set_ylabel(name); ax.set_title(f'EKF vs GT R2={r2(fused[:nf, i], fGT[:, i]):.2f}', fontsize=8, loc='right')
        if i == 0: ax.legend(fontsize=8, loc='upper right')
    axes[-1].set_xlabel('t (s, relative)')
    fig.suptitle('EKF FUSED target-relative flow [h_tr; w] vs corner vs GT', fontsize=12); plt.show()
    if tvel.ndim == 2 and len(tvel) > 0:
        nt = min(len(tvel), len(ti))
        fig, ax = plt.subplots(1, 1, figsize=(12, 3.2), constrained_layout=True)
        for k, lab in enumerate(['x', 'y', 'z']): ax.plot(ti[:nt] - ti[0], tvel[:nt, k], label=f'h_tv_{lab}')
        ax.axhline(0, color='k', ls=':', lw=0.8); ax.set_xlabel('t (s)'); ax.set_ylabel('h_tv (flow units)')
        ax.set_title('EKF target velocity h_tv  (~0 stationary; = rover velocity if moving)')
        ax.legend(fontsize=8); plt.show()

## 2 — Landing-to-touchdown validation

Runs `validate_flow_to_touchdown.validate()` on the landing run: ring R²/slope binned by
**altitude** into the final 0.0–0.2 m, marker→ring switch continuity, and the ω sign-check.
This is the cal in its real regime.

In [ ]:
if LANDING_RUN:
    vft.validate(LANDING_RUN)
else:
    print('No landing run found — record one to validation_data/landing/ or test_data/.')